In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

X = pd.read_parquet("../data/processed/luad_X.parquet")
labels = pd.read_csv("../data/processed/luad_labels.csv")

y = labels["label"]

(X.index == labels["sample_id"]).all(), X.shape, labels["sample_type"].value_counts()

(np.True_,
 (574, 20530),
 sample_type
 tumor     515
 normal     59
 Name: count, dtype: int64)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape, y_train.value_counts(), y_test.value_counts()

((459, 20530),
 (115, 20530),
 label
 1.0    412
 0.0     47
 Name: count, dtype: int64,
 label
 1.0    103
 0.0     12
 Name: count, dtype: int64)

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
selector_model = LogisticRegression(
    penalty="l1",
    solver="saga",
    class_weight="balanced",
    max_iter=10000,
    C=0.1,
    random_state=42,
)

selector_model.fit(X_train_scaled, y_train)

/Users/adityaanand/dev/cancer-lab/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/adityaanand/dev/cancer-lab/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'l1'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Ta

In [11]:
coefs = selector_model.coef_[0]

ranking = pd.DataFrame({
    "gene": X.columns,
    "coef": coefs,
    "abs_coef": np.abs(coefs),
})

ranking = ranking.sort_values("abs_coef", ascending=False)

ranking.head(30)

,gene,coef,abs_coef
2939,PYCR1,0.355015,0.355015
20363,ETV4,0.244203,0.244203
10326,CD5L,-0.224410,0.224410
9724,ADM2,0.202896,0.202896
2677,LGR4,0.177042,0.177042
7081,ATP10B,0.152263,0.152263
7970,OR6K3,-0.150351,0.150351
14020,EMP2,-0.147496,0.147496
11727,SPP1,0.108235,0.108235
16893,C10orf67,-0.107084,0.107084


In [12]:
panel_sizes = [20, 10, 5, 3, 2]

def evaluate_panel(selected_genes):
    X_train_panel = X_train[selected_genes]
    X_test_panel = X_test[selected_genes]

    scaler_panel = StandardScaler()
    X_train_panel_scaled = scaler_panel.fit_transform(X_train_panel)
    X_test_panel_scaled = scaler_panel.transform(X_test_panel)

    model = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42,
    )

    model.fit(X_train_panel_scaled, y_train)

    y_pred = model.predict(X_test_panel_scaled)
    y_prob = model.predict_proba(X_test_panel_scaled)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    }

In [13]:
panel_results = []
panel_genes = []

for k in panel_sizes:
    selected_genes = ranking.head(k)["gene"].tolist()
    metrics = evaluate_panel(selected_genes)

    panel_results.append({
        "panel_size": k,
        **metrics,
    })

    for rank, gene in enumerate(selected_genes, start=1):
        panel_genes.append({
            "panel_size": k,
            "rank": rank,
            "gene": gene,
            "coef": ranking.loc[ranking["gene"] == gene, "coef"].iloc[0],
            "abs_coef": ranking.loc[ranking["gene"] == gene, "abs_coef"].iloc[0],
        })

panel_results_df = pd.DataFrame(panel_results)
panel_genes_df = pd.DataFrame(panel_genes)

panel_results_df

,panel_size,accuracy,balanced_accuracy,f1,roc_auc,confusion_matrix
0,20,1.000000,1.000000,1.000000,1.000000,"[[12, 0], [0, 103]]"
1,10,0.991304,0.995146,0.995122,1.000000,"[[12, 0], [1, 102]]"
2,5,0.991304,0.995146,0.995122,0.999191,"[[12, 0], [1, 102]]"
3,3,0.982609,0.990291,0.990196,1.000000,"[[12, 0], [2, 101]]"
4,2,0.956522,0.975728,0.975124,0.993528,"[[12, 0], [5, 98]]"


In [14]:
panel_genes_df

,panel_size,rank,gene,coef,abs_coef
0,20,1,PYCR1,0.355015,0.355015
1,20,2,ETV4,0.244203,0.244203
2,20,3,CD5L,-0.224410,0.224410
3,20,4,ADM2,0.202896,0.202896
4,20,5,LGR4,0.177042,0.177042
5,20,6,ATP10B,0.152263,0.152263
6,20,7,OR6K3,-0.150351,0.150351
7,20,8,EMP2,-0.147496,0.147496
8,20,9,SPP1,0.108235,0.108235
9,20,10,C10orf67,-0.107084,0.107084


In [15]:
panel_results_df.to_csv(
    "../data/processed/unconstrained_panel_results.csv",
    index=False,
)

panel_genes_df.to_csv(
    "../data/processed/unconstrained_panel_genes.csv",
    index=False,
)

### Bulk expression audit of unconstrained selected genes

In [16]:
selected_gene_list = panel_genes_df["gene"].drop_duplicates().tolist()

selected_gene_list

['PYCR1',
 'ETV4',
 'CD5L',
 'ADM2',
 'LGR4',
 'ATP10B',
 'OR6K3',
 'EMP2',
 'SPP1',
 'C10orf67',
 'PLEKHN1',
 'FAM83A',
 'CELA2B',
 'CYP3A7',
 'EPB42',
 'RTKN2',
 'ANGPT4',
 'NCRNA00115',
 'HBM',
 'OVCH1']

In [17]:
X_labeled = X.copy()
X_labeled["sample_type"] = labels["sample_type"].values
X_labeled["label"] = labels["label"].values

tumor_mean = X_labeled[X_labeled["sample_type"] == "tumor"][selected_gene_list].mean()
normal_mean = X_labeled[X_labeled["sample_type"] == "normal"][selected_gene_list].mean()

bulk_audit = pd.DataFrame({
    "gene": selected_gene_list,
    "tumor_mean": [tumor_mean[g] for g in selected_gene_list],
    "normal_mean": [normal_mean[g] for g in selected_gene_list],
})

bulk_audit["tumor_minus_normal"] = (
    bulk_audit["tumor_mean"] - bulk_audit["normal_mean"]
)

bulk_audit

,gene,tumor_mean,normal_mean,tumor_minus_normal
0,PYCR1,11.354052,7.682817,3.671235
1,ETV4,9.540134,5.769737,3.770396
2,CD5L,1.302915,5.345639,-4.042724
3,ADM2,8.139807,4.753285,3.386523
4,LGR4,10.222406,7.694939,2.527467
5,ATP10B,6.659753,2.178944,4.480809
6,OR6K3,0.188966,1.937473,-1.748507
7,EMP2,12.107239,15.026341,-2.919102
8,SPP1,12.460326,7.720673,4.739654
9,C10orf67,1.686859,6.395881,-4.709022


In [18]:
bulk_audit["direction"] = np.where(
    bulk_audit["tumor_minus_normal"] > 0,
    "tumor_up",
    "normal_up",
)

bulk_audit.sort_values("tumor_minus_normal", ascending=False)

,gene,tumor_mean,normal_mean,tumor_minus_normal,direction
11,FAM83A,10.108718,3.877571,6.231147,tumor_up
8,SPP1,12.460326,7.720673,4.739654,tumor_up
5,ATP10B,6.659753,2.178944,4.480809,tumor_up
1,ETV4,9.540134,5.769737,3.770396,tumor_up
0,PYCR1,11.354052,7.682817,3.671235,tumor_up
3,ADM2,8.139807,4.753285,3.386523,tumor_up
10,PLEKHN1,6.662193,3.333259,3.328934,tumor_up
4,LGR4,10.222406,7.694939,2.527467,tumor_up
17,NCRNA00115,4.647448,3.419988,1.227460,tumor_up
14,EPB42,0.157139,1.466269,-1.309130,normal_up


In [19]:
gene_best_rank = (
    panel_genes_df
    .groupby("gene")["rank"]
    .min()
    .reset_index()
    .rename(columns={"rank": "best_rank"})
)

gene_max_abscoef = (
    panel_genes_df
    .groupby("gene")["abs_coef"]
    .max()
    .reset_index()
)

bulk_audit = (
    bulk_audit
    .merge(gene_best_rank, on="gene", how="left")
    .merge(gene_max_abscoef, on="gene", how="left")
)

bulk_audit.sort_values("best_rank")

,gene,tumor_mean,normal_mean,tumor_minus_normal,direction,best_rank,abs_coef
0,PYCR1,11.354052,7.682817,3.671235,tumor_up,1,0.355015
1,ETV4,9.540134,5.769737,3.770396,tumor_up,2,0.244203
2,CD5L,1.302915,5.345639,-4.042724,normal_up,3,0.224410
3,ADM2,8.139807,4.753285,3.386523,tumor_up,4,0.202896
4,LGR4,10.222406,7.694939,2.527467,tumor_up,5,0.177042
5,ATP10B,6.659753,2.178944,4.480809,tumor_up,6,0.152263
6,OR6K3,0.188966,1.937473,-1.748507,normal_up,7,0.150351
7,EMP2,12.107239,15.026341,-2.919102,normal_up,8,0.147496
8,SPP1,12.460326,7.720673,4.739654,tumor_up,9,0.108235
9,C10orf67,1.686859,6.395881,-4.709022,normal_up,10,0.107084


In [20]:
bulk_audit.to_csv(
    "../data/processed/unconstrained_panel_bulk_audit.csv",
    index=False,
)